# Notebook 1/3 — Foundation: Frozen Bank, Teacher Spot-Check, Weight-Transfer Equivalence

**এই notebook train করে না — শুধু verify করে যে Day 2-এর pruning experiment শুরু করার ভিত্তি ঠিক আছে।**

সবকিছু ORIGINAL repo code দিয়ে করা হয়েছে (`DL_DOA/src/*.py` সরাসরি import, reimplement না করে) — যাতে metric-এ কোনো drift না হয়।

### ৪টা জিনিস verify হবে
1. **Teacher spot-check** — pretrained ResNet-এর real Pd/RMSE (paper graph থেকে পড়া না, নিজে চালিয়ে) — original evaluator দিয়ে
2. **Weight-transfer equivalence** — pruning pipeline (r=12, কিছুই না ফেলে) দিয়ে copy করলে prediction হুবহু মেলে কিনা
3. **Per-step timing** — Day 2-7 বাজেট করার জন্য real measurement (অনুমান না)
4. **Calibration bank** load — Day 2-এর Taylor-importance scoring-এর জন্য ready রাখা

### Kaggle-এ যা attach করতে হবে (Add Data)
- **DL_DOA_CLONE repo** (GitHub) — `DL_DOA/src/*.py` আর pretrained weight পাওয়ার জন্য
- **frozen_banks folder** (তুমি upload করবে) — `eval_bank.npz` (19 MB) + `calibration_bank.npz` (5 MB)

In [ ]:
# Cell 1 — Setup
# NOTE: if you hit ResourceExhaustedError on a TINY tensor (a few MB) after a
# previous OOM crash in this same kernel, restarting the kernel/session fixes
# it -- a crashed allocation can leave GPU memory fragmented in a way that
# clear_session() alone cannot undo. Restart, then Run All from here.

import importlib, subprocess, sys
try:
    import cv2
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python-headless'], check=True)

import os, time, json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, Input, BatchNormalization, Activation, Add, Conv2DTranspose
from tensorflow.keras.models import Model

tf.keras.backend.clear_session()
tf.get_logger().setLevel('ERROR')
np.random.seed(42); tf.random.set_seed(42)

gpus = tf.config.list_physical_devices('GPU')
print(f'TF: {tf.__version__}  |  GPU: {gpus}')
for g in gpus: tf.config.experimental.set_memory_growth(g, True)

for i in range(len(gpus)):
    try:
        info = tf.config.experimental.get_memory_info(f'GPU:{i}')
        print(f'  GPU:{i} memory -- current: {info[\"current\"]/1e6:.0f} MB, peak: {info[\"peak\"]/1e6:.0f} MB')
    except Exception as e:
        print(f'  GPU:{i} memory info unavailable ({e})')

OUT_DIR = '/kaggle/working'
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# Cell 2 — Locate repo source, pretrained weights, and frozen banks (search, don't assume paths)
def find_path(name_pattern, is_dir=False):
    from pathlib import Path
    for root in ['/kaggle/input', '/kaggle/working', '.', '/content']:
        if not os.path.isdir(root): continue
        for p in Path(root).rglob(name_pattern):
            if is_dir and p.is_dir(): return str(p)
            if not is_dir and p.is_file(): return str(p)
    return None

# Repo source: find DL_DOA/src directory (contains tvt_models.py, TVT_Blob_Inference.py, tvt_data_generation_v3.py)
SRC_MODEL_FILE = find_path('tvt_models.py')
assert SRC_MODEL_FILE is not None, 'DL_DOA_CLONE repo not found -- Add Data and attach it'
DL_DOA_DIR = os.path.dirname(os.path.dirname(SRC_MODEL_FILE))   # parent of src/
print(f'DL_DOA dir: {DL_DOA_DIR}')

WEIGHTS_PATH = find_path('inf_model_007_256_resnet.h5')
assert WEIGHTS_PATH is not None, 'pretrained weights not found'
print(f'Weights: {WEIGHTS_PATH}')

EVAL_BANK_PATH = find_path('eval_bank.npz')
CALIB_BANK_PATH = find_path('calibration_bank.npz')
assert EVAL_BANK_PATH is not None, 'eval_bank.npz not found -- attach the frozen_banks dataset'
assert CALIB_BANK_PATH is not None, 'calibration_bank.npz not found -- attach the frozen_banks dataset'
print(f'Eval bank: {EVAL_BANK_PATH}')
print(f'Calibration bank: {CALIB_BANK_PATH}')

In [ ]:
# Cell 3 — Import ORIGINAL modules directly (no reimplementation -- avoids metric drift)
sys.path.insert(0, DL_DOA_DIR)

from src.tvt_models import Resnet
from src.TVT_Blob_Inference import (
    get_blob_detector, get_blob_peaks, peaks_to_angles,
    prepare_for_metric, get_ang_difference, filter_angles,
)

print('✅ Imported Resnet + original evaluator functions from the repo, unmodified')

In [ ]:
# Cell 4 — Load frozen banks (generated locally, NOT regenerated here -- reproducibility across all 3 notebooks)
eval_bank = np.load(EVAL_BANK_PATH)
EVAL_DATA, EVAL_FEAT, EVAL_META = eval_bank['data'], eval_bank['feat'], eval_bank['meta']
SIGMA = float(eval_bank['sigma']); M = int(eval_bank['M'])

calib_bank = np.load(CALIB_BANK_PATH)
CALIB_DATA, CALIB_GT, CALIB_FEAT, CALIB_META = (
    calib_bank['data'], calib_bank['gt'], calib_bank['feat'], calib_bank['meta']
)

print(f'Eval bank:        {EVAL_DATA.shape[0]} samples,  sigma={SIGMA}, M={M}')
snrs, counts = np.unique(EVAL_META[:,1], return_counts=True)
print(f'  SNR distribution: {dict(zip(snrs.tolist(), counts.tolist()))}')
print(f'Calibration bank:  {CALIB_DATA.shape[0]} samples (with GT, for Day 2 Taylor scoring)')

In [ ]:
# Cell 5 — Load pretrained teacher (built with the ORIGINAL Resnet() constructor)
teacher = Resnet(input_shape=(64, 64, 2))
teacher.load_weights(WEIGHTS_PATH)
TEACHER_PARAMS = teacher.count_params()
print(f'✅ Teacher loaded: {TEACHER_PARAMS:,} params')

In [ ]:
# Cell 6 — DIAGNOSTIC 1: Teacher spot-check on the frozen eval bank
# Reuses the original evaluator's per-sample logic verbatim, just iterating over
# OUR saved fixed samples instead of the live generator -- this is the number
# every later student model gets compared against, computed ourselves (not the
# hardcoded paper-graph values used earlier in this project).
#
# batch_size kept small (8) -- the 64-block ResNet at 128x128 internal resolution
# is memory-heavy per sample; a batch of 64 OOM'd on the Kaggle T4. Inference
# speed doesn't matter much here (blob detection on CPU dominates the loop anyway).

def evaluate_on_bank(model, data_arr, feat_arr, meta_arr, batch_size=8, max_deg_error=1.0):
    detector = get_blob_detector()
    results_by_snr = {}
    N = data_arr.shape[0]
    t0 = time.time()
    for start in range(0, N, batch_size):
        end = min(start + batch_size, N)
        preds = model(data_arr[start:end], training=False)
        for j in range(end - start):
            idx = start + j
            L = int(meta_arr[idx, 0]); snr = int(meta_arr[idx, 1])
            peaks, amps = get_blob_peaks(preds[j], detector)
            order = np.argsort(-amps)
            peaks = peaks[order[:L]]
            angles_est = peaks_to_angles(peaks, sigma=SIGMA, grid_size=M)
            gt_angles, pred_angles = prepare_for_metric(angles_est, feat_arr[idx])
            results_by_snr.setdefault(snr, []).append((gt_angles, pred_angles))
        if (start // batch_size) % 200 == 0:
            print(f'  {end}/{N}  ({time.time()-t0:.1f}s)')

    final_pd, final_rmse = {}, {}
    for snr, examples in results_by_snr.items():
        good_all, bad_all = [], []
        for gt, pred in examples:
            if np.isnan(pred).any(): continue
            diffs = get_ang_difference(gt, pred)
            good, bad = filter_angles(diffs, max_deg_error=max_deg_error)
            good_all.append(good); bad_all.append(bad)
        good_all = np.concatenate(good_all) if good_all else np.array([])
        bad_all = np.concatenate(bad_all) if bad_all else np.array([])
        total = len(good_all) + len(bad_all)
        final_pd[snr] = len(good_all) / total if total > 0 else np.nan
        final_rmse[snr] = np.sqrt(np.mean(good_all ** 2)) if len(good_all) > 0 else np.nan
    return final_pd, final_rmse

print('Running teacher spot-check on frozen eval bank (8000 samples, original evaluator)...')
teacher_pd, teacher_rmse = evaluate_on_bank(teacher, EVAL_DATA, EVAL_FEAT, EVAL_META, batch_size=8)
print('\n✅ Teacher spot-check complete')

In [ ]:
# Cell 7 — Visual: our recomputed teacher Pd/RMSE vs paper Table II reference
REF_PD   = {-10:0.204, -5:0.437, 0:0.643, 5:0.779, 10:0.862, 15:0.899, 20:0.925, 25:0.938}
REF_RMSE = {-10:0.553, -5:0.512, 0:0.458, 5:0.392, 10:0.326, 15:0.279, 20:0.253, 25:0.238}

snrs_sorted = sorted(teacher_pd.keys())
print('='*70)
print(f'{"SNR":>5} | {"Our Pd":>8} {"Paper Pd":>9} {"ΔPd":>8} | {"Our RMSE":>9} {"Paper RMSE":>11}')
print('-'*70)
max_abs_dpd = 0
for s in snrs_sorted:
    dpd = teacher_pd[s] - REF_PD[s]
    max_abs_dpd = max(max_abs_dpd, abs(dpd))
    print(f'{s:>5} | {teacher_pd[s]:>8.4f} {REF_PD[s]:>9.3f} {dpd:>+8.4f} | '
          f'{teacher_rmse[s]:>9.4f} {REF_RMSE[s]:>11.3f}')
print('='*70)
print(f'Max |ΔPd| across SNR: {max_abs_dpd:.4f}  '
      f'({"within expected noise" if max_abs_dpd < 0.03 else "⚠️ investigate before Day 2"})')

fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
axs[0].plot(snrs_sorted, [teacher_pd[s] for s in snrs_sorted], 'o-', label='Ours (frozen bank)')
axs[0].plot(snrs_sorted, [REF_PD[s] for s in snrs_sorted], 's--', label='Paper Table II')
axs[0].set_xlabel('SNR (dB)'); axs[0].set_ylabel('Pd'); axs[0].set_title('Teacher Pd'); axs[0].legend(); axs[0].grid(alpha=.3)

axs[1].plot(snrs_sorted, [teacher_rmse[s] for s in snrs_sorted], 'o-', label='Ours (frozen bank)')
axs[1].plot(snrs_sorted, [REF_RMSE[s] for s in snrs_sorted], 's--', label='Paper Table II')
axs[1].set_xlabel('SNR (dB)'); axs[1].set_ylabel('RMSE (deg)'); axs[1].set_title('Teacher RMSE'); axs[1].legend(); axs[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'teacher_spotcheck.png'), dpi=130)
plt.show()

In [ ]:
# Cell 8 — Pruned-ResNet builder (parametrized internal width r) + weight-copy function
# Architecture spec: 64 blocks, each receives/returns 12 channels, internal width r.

def res_conv_pruned(x, r, out_filters=12):
    skip = x
    x = Conv2D(r, 5, padding='same')(x); x = BatchNormalization()(x); x = Activation('relu')(x)
    x = Conv2D(out_filters, 5, padding='same')(x); x = BatchNormalization()(x)
    x = Add()([x, skip]); x = Activation('relu')(x)
    return x

def build_pruned_resnet(r, n_blocks=64, input_shape=(64, 64, 2)):
    x_in = Input(shape=input_shape)
    x = Conv2DTranspose(12, (5, 5), strides=(2, 2), padding='same')(x_in)
    for _ in range(n_blocks):
        x = res_conv_pruned(x, r)
    x = Conv2DTranspose(1, (5, 5), strides=(2, 2), padding='same')(x)
    return Model(x_in, x, name=f'PrunedResNet-r{r}')

def get_weighted_layers(model):
    return [l for l in model.layers if l.get_weights()]

def copy_weights_to_pruned(teacher_model, student_model, filter_indices_per_block, n_blocks=64):
    """filter_indices_per_block: list of n_blocks arrays, each of length r (which of
    the 12 original bottleneck channels to keep). Steps follow the doc's section 5.3."""
    tw = get_weighted_layers(teacher_model)
    sw = get_weighted_layers(student_model)
    assert len(tw) == len(sw) == (1 + 4 * n_blocks + 1), f'{len(tw)} vs {len(sw)} weighted layers'

    sw[0].set_weights(tw[0].get_weights())   # initial transpose -- unchanged

    for i in range(n_blocks):
        J = np.asarray(filter_indices_per_block[i])
        base = 1 + 4 * i
        t_conv1, t_bn1, t_conv2, t_bn2 = tw[base:base+4]
        s_conv1, s_bn1, s_conv2, s_bn2 = sw[base:base+4]

        k, b = t_conv1.get_weights()
        s_conv1.set_weights([k[:, :, :, J], b[J]])          # Conv1: slice OUTPUT channels

        gamma, beta, mean, var = t_bn1.get_weights()
        s_bn1.set_weights([gamma[J], beta[J], mean[J], var[J]])  # BN1: slice all 4

        k2, b2 = t_conv2.get_weights()
        s_conv2.set_weights([k2[:, J, :], b2])               # Conv2: slice INPUT channels only

        s_bn2.set_weights(t_bn2.get_weights())                # BN2: unchanged

    sw[-1].set_weights(tw[-1].get_weights())  # final transpose -- unchanged

print('✅ Pruned-ResNet builder + weight-copy function ready')

In [ ]:
# Cell 9 — DIAGNOSTIC 2: weight-transfer equivalence check at r=12 (identity -- nothing pruned)
# If this doesn't reproduce the teacher exactly, the copy pipeline has a bug and
# pruning must NOT proceed until it's fixed.
# batch kept small (8) -- same OOM reason as Cell 6.

student_r12 = build_pruned_resnet(r=12)
identity_indices = [list(range(12))] * 64
copy_weights_to_pruned(teacher, student_r12, identity_indices)

sample_batch = EVAL_DATA[:8]
pred_teacher = teacher(sample_batch, training=False).numpy()
pred_student = student_r12(sample_batch, training=False).numpy()

max_abs_diff = np.max(np.abs(pred_teacher - pred_student))
mean_abs_diff = np.mean(np.abs(pred_teacher - pred_student))
TOL = 1e-4
print(f'Max  |teacher - rebuilt| : {max_abs_diff:.3e}')
print(f'Mean |teacher - rebuilt|: {mean_abs_diff:.3e}')
print(f'Tolerance: {TOL:.0e}')
print(f'{"✅ PASS" if max_abs_diff < TOL else "❌ FAIL -- fix copy_weights_to_pruned before Day 2"}')

fig, ax = plt.subplots(figsize=(5, 5))
idx_pts = np.random.choice(pred_teacher.size, min(20000, pred_teacher.size), replace=False)
ax.scatter(pred_teacher.ravel()[idx_pts], pred_student.ravel()[idx_pts], s=2, alpha=0.3)
lims = [pred_teacher.min(), pred_teacher.max()]
ax.plot(lims, lims, 'r--', lw=1, label='y = x (perfect match)')
ax.set_xlabel('Teacher prediction'); ax.set_ylabel('Rebuilt (r=12) prediction')
ax.set_title('Weight-transfer equivalence check'); ax.legend()
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'weight_equivalence.png'), dpi=130); plt.show()

In [ ]:
# Cell 10 — DIAGNOSTIC 3: per-step timing (r=12 vs r=8), for real Day 2-7 budgeting
# batch kept small (8) -- training=True + GradientTape needs more memory than
# inference alone (activations kept for backprop), so this is even more OOM-prone
# than Cell 6/9 at the same batch size.

def measure_step_time(model, n_warmup=5, n_measure=20, batch=8):
    opt = tf.keras.optimizers.Adam(1e-4)
    x = tf.random.normal((batch, 64, 64, 2))
    y = tf.random.normal((batch, 256, 256, 1))

    @tf.function
    def step():
        with tf.GradientTape() as tape:
            pred = model(x, training=True)
            loss = tf.reduce_mean(tf.square(pred - y))
        grads = tape.gradient(loss, model.trainable_variables)
        opt.apply_gradients(zip(grads, model.trainable_variables))
        return loss

    for _ in range(n_warmup): step()
    t0 = time.time()
    for _ in range(n_measure): step()
    return (time.time() - t0) / n_measure

TRAIN_BATCH = 8   # actual training batch size to use in Notebook 2 -- keep consistent
STEPS_PER_EPOCH = int(np.ceil(10000 / TRAIN_BATCH))  # was 312 at batch=32; recompute for the new batch

student_r8 = build_pruned_resnet(r=8)   # placeholder identity selection -- timing only, not real selection

sec_per_step_r12 = measure_step_time(student_r12, batch=TRAIN_BATCH)
sec_per_step_r8  = measure_step_time(student_r8, batch=TRAIN_BATCH)

print(f'Batch size used for timing (and for Notebook 2 training): {TRAIN_BATCH}')
print(f'r=12 ({student_r12.count_params():,} params):  {sec_per_step_r12*1000:.1f} ms/step')
print(f'r=8  ({student_r8.count_params():,} params):   {sec_per_step_r8*1000:.1f} ms/step')
print()
for epochs in [50, 150, 300, 500]:
    total_r12 = sec_per_step_r12 * STEPS_PER_EPOCH * epochs / 3600
    total_r8  = sec_per_step_r8  * STEPS_PER_EPOCH * epochs / 3600
    print(f'{epochs:>4} epochs x {STEPS_PER_EPOCH} steps  ->  r=12: {total_r12:.2f}h   r=8: {total_r8:.2f}h')

print()
print('এই real numbers দিয়ে Day 2-7-এর epoch budget ঠিক করো -- অনুমান না করে।')
print(f'(batch={TRAIN_BATCH} ছোট রাখা হয়েছে OOM এড়াতে -- Notebook 2-তেও এই batch size ব্যবহার করো)')

In [ ]:
# Cell 11 — Day 1 diagnostic summary (go/no-go before Day 2)
checks = []
checks.append(('Repo source + original evaluator imported', True))
checks.append(('Frozen eval bank loaded (8000 samples, 8 SNR points)', EVAL_DATA.shape[0] == 8000))
checks.append(('Calibration bank loaded (600 samples, with GT)', CALIB_DATA.shape[0] == 600))
checks.append((f'Teacher spot-check close to paper (max |ΔPd|={max_abs_dpd:.4f} < 0.03)', max_abs_dpd < 0.03))
checks.append((f'Weight-transfer equivalence (max diff={max_abs_diff:.1e} < {TOL:.0e})', max_abs_diff < TOL))

print('='*70)
print('DAY 1 DIAGNOSTIC SUMMARY')
print('='*70)
all_pass = True
for desc, ok in checks:
    print(f'  [{"PASS" if ok else "FAIL"}]  {desc}')
    all_pass = all_pass and ok
print('='*70)
print(f'{"✅ ALL CHECKS PASSED -- proceed to Notebook 2 (pruning + fine-tuning)" if all_pass else "❌ FIX FAILING CHECKS BEFORE NOTEBOOK 2"}')

results = {
    'teacher_params': TEACHER_PARAMS,
    'teacher_pd': {str(k): float(v) for k, v in teacher_pd.items()},
    'teacher_rmse': {str(k): float(v) for k, v in teacher_rmse.items()},
    'max_abs_dpd_vs_paper': float(max_abs_dpd),
    'weight_equivalence_max_diff': float(max_abs_diff),
    'sec_per_step_r12': float(sec_per_step_r12),
    'sec_per_step_r8': float(sec_per_step_r8),
    'all_checks_passed': bool(all_pass),
}
with open(os.path.join(OUT_DIR, 'day1_foundation_results.json'), 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nSaved: day1_foundation_results.json  (Notebook 2 will load this + reuse the frozen banks)')